# OmicsDRP — 5-fold CV results summary

Aggregates the nested-CV outer-test metrics (one value per fold) into a table of
**mean / std / median** (plus min/max) per condition, for each split regime
(`mixed`, `unseen_cell`, `unseen_drug`).

Reads each experiment's `folds/fold_k_test_metrics.json` directly, so partially
finished sweeps are included (with their current fold count).

Key metrics: **RMSE, MAE** (lower = better) and **R², Pearson, Spearman**
(higher = better).

In [1]:
import json, glob, os
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

def find_results_dir(start=None):
    start = Path(start or os.getcwd()).resolve()
    for base in [start, *start.parents]:
        for rel in ['scripts/Results', 'omicsdrp/scripts/Results', 'Results']:
            p = base / rel
            if p.is_dir():
                return str(p)
    raise FileNotFoundError('Results dir not found from ' + str(start))

RESULTS_DIR = find_results_dir()
print('RESULTS_DIR =', RESULTS_DIR)

RESULTS_DIR = /project/OmicsDRP_Review/omicsdrp/scripts/Results


## Load per-fold outer-test metrics

In [2]:
METRICS = ['rmse', 'mae', 'r2', 'pearson', 'spearman']
OMICS_ORDER = ['SNP', 'MET', 'CNV', 'RNA']

def load_fold_records(results_dir):
    rows = []
    for cfg_path in sorted(glob.glob(os.path.join(results_dir, '*', 'config.json'))):
        d = os.path.dirname(cfg_path)
        fold_jsons = sorted(glob.glob(os.path.join(d, 'folds', 'fold_*_test_metrics.json')))
        if not fold_jsons:
            continue  # no completed folds yet
        cfg = json.load(open(cfg_path))
        omics_str = '+'.join([o for o in OMICS_ORDER if o in cfg['omics']])
        for fj in fold_jsons:
            r = json.load(open(fj))
            rec = {'split_mode': cfg['split_mode'], 'omics': omics_str,
                   'cell': cfg['cell_encoder'], 'drug': cfg['drug_encoder'],
                   'tag': os.path.basename(d), 'fold': int(r.get('fold', -1)),
                   'best_epoch': r.get('best_epoch')}
            for m in METRICS:
                rec[m] = r.get(m, np.nan)
            rows.append(rec)
    return pd.DataFrame(rows)

folds = load_fold_records(RESULTS_DIR)
print(f"{folds['tag'].nunique()} experiments, {len(folds)} fold-records")
folds.groupby('split_mode')['tag'].nunique().rename('experiments')

13 experiments, 64 fold-records


split_mode
mixed          12
unseen_cell     1
Name: experiments, dtype: int64

## Aggregate per condition
Condition = (`split_mode`, `omics`, `cell`, `drug`). Stats over the fold values.

In [3]:
COND = ['split_mode', 'omics', 'cell', 'drug']

agg = folds.groupby(COND)[METRICS].agg(['mean', 'std', 'median', 'min', 'max'])
agg.columns = [f'{m}_{s}' for m, s in agg.columns]
agg = agg.join(folds.groupby(COND)['fold'].nunique().rename('n_folds')).reset_index()
agg = agg.sort_values(['split_mode', 'rmse_mean']).reset_index(drop=True)
print('conditions:', len(agg))
agg.round(4)

conditions: 13


,split_mode,omics,cell,drug,rmse_mean,rmse_std,rmse_median,rmse_min,rmse_max,mae_mean,mae_std,mae_median,mae_min,mae_max,r2_mean,r2_std,r2_median,r2_min,r2_max,pearson_mean,pearson_std,pearson_median,pearson_min,pearson_max,spearman_mean,spearman_std,spearman_median,spearman_min,spearman_max,n_folds
0,mixed,SNP+MET+CNV+RNA,attention,morgan,0.9652,0.0235,0.9644,0.9380,0.9933,0.7156,0.0177,0.7191,0.6965,0.7332,0.8834,0.0065,0.8848,0.8750,0.8900,0.9401,0.0035,0.9411,0.9357,0.9435,0.9143,0.0038,0.9159,0.9101,0.9182,5
1,mixed,CNV+RNA,attention,morgan,0.9662,0.0116,0.9717,0.9531,0.9762,0.7160,0.0088,0.7183,0.7067,0.7249,0.8832,0.0038,0.8820,0.8794,0.8873,0.9399,0.0020,0.9393,0.9378,0.9420,0.9139,0.0019,0.9130,0.9123,0.9169,5
2,mixed,SNP+MET+CNV+RNA,attention,unimol,0.9671,0.0118,0.9621,0.9582,0.9874,0.7187,0.0099,0.7188,0.7048,0.7329,0.8830,0.0033,0.8843,0.8773,0.8853,0.9399,0.0019,0.9406,0.9367,0.9413,0.9139,0.0025,0.9149,0.9096,0.9158,5
3,mixed,SNP+MET+CNV+RNA,attention,molformer,0.9688,0.0083,0.9646,0.9617,0.9824,0.7171,0.0060,0.7179,0.7107,0.7239,0.8826,0.0029,0.8836,0.8778,0.8848,0.9398,0.0015,0.9404,0.9371,0.9408,0.9139,0.0017,0.9141,0.9112,0.9158,5
4,mixed,SNP+MET+CNV+RNA,attention,chemberta,0.9693,0.0150,0.9669,0.9531,0.9929,0.7216,0.0104,0.7187,0.7087,0.7361,0.8824,0.0046,0.8838,0.8751,0.8865,0.9396,0.0024,0.9403,0.9356,0.9415,0.9134,0.0027,0.9131,0.9093,0.9161,5
5,mixed,MET+RNA,attention,morgan,0.9700,0.0122,0.9640,0.9589,0.9850,0.7206,0.0052,0.7182,0.7151,0.7269,0.8822,0.0040,0.8846,0.8771,0.8858,0.9395,0.0022,0.9408,0.9367,0.9415,0.9129,0.0022,0.9141,0.9105,0.9148,5
6,mixed,SNP+MET+RNA,attention,morgan,0.9701,0.0049,0.9701,0.9632,0.9764,0.7201,0.0080,0.7234,0.7098,0.7282,0.8823,0.0011,0.8830,0.8809,0.8832,0.9394,0.0006,0.9397,0.9386,0.9399,0.9132,0.0011,0.9138,0.9120,0.9142,5
7,mixed,SNP+RNA,attention,morgan,0.9709,0.0100,0.9705,0.9601,0.9813,0.7183,0.0038,0.7177,0.7125,0.7220,0.8820,0.0034,0.8830,0.8782,0.8859,0.9393,0.0018,0.9399,0.9373,0.9413,0.9131,0.0019,0.9121,0.9116,0.9159,5
8,mixed,SNP+MET+CNV+RNA,attention,graphormer,0.9722,0.0024,0.9713,0.9694,0.9756,0.7218,0.0039,0.7213,0.7168,0.7268,0.8817,0.0015,0.8822,0.8794,0.8832,0.9393,0.0009,0.9397,0.9379,0.9400,0.9131,0.0007,0.9131,0.9120,0.9138,5
9,mixed,SNP+CNV+RNA,attention,morgan,0.9739,0.0144,0.9742,0.9602,0.9961,0.7235,0.0119,0.7202,0.7139,0.7429,0.8813,0.0042,0.8825,0.8751,0.8850,0.9390,0.0022,0.9397,0.9357,0.9409,0.9126,0.0026,0.9137,0.9082,0.9146,5


## Readable tables per split regime
`mean ± std (median)` for each metric, sorted by RMSE (best first).

In [4]:
def pretty(sub):
    o = sub[['omics', 'cell', 'drug']].reset_index(drop=True).copy()
    for m in METRICS:
        o[m] = [f"{r[f'{m}_mean']:.4f} ± {r[f'{m}_std']:.4f} (med {r[f'{m}_median']:.4f})"
                for _, r in sub.reset_index(drop=True).iterrows()]
    o['n_folds'] = sub['n_folds'].values
    return o

for sm in ['mixed', 'unseen_cell', 'unseen_drug']:
    sub = agg[agg['split_mode'] == sm].sort_values('rmse_mean')
    if len(sub) == 0:
        continue
    print(f'\n===== split_mode = {sm}  (sorted by RMSE, lower = better) =====')
    display(pretty(sub))


===== split_mode = mixed  (sorted by RMSE, lower = better) =====


,omics,cell,drug,rmse,mae,r2,pearson,spearman,n_folds
0,SNP+MET+CNV+RNA,attention,morgan,0.9652 ± 0.0235 (med 0.9644),0.7156 ± 0.0177 (med 0.7191),0.8834 ± 0.0065 (med 0.8848),0.9401 ± 0.0035 (med 0.9411),0.9143 ± 0.0038 (med 0.9159),5
1,CNV+RNA,attention,morgan,0.9662 ± 0.0116 (med 0.9717),0.7160 ± 0.0088 (med 0.7183),0.8832 ± 0.0038 (med 0.8820),0.9399 ± 0.0020 (med 0.9393),0.9139 ± 0.0019 (med 0.9130),5
2,SNP+MET+CNV+RNA,attention,unimol,0.9671 ± 0.0118 (med 0.9621),0.7187 ± 0.0099 (med 0.7188),0.8830 ± 0.0033 (med 0.8843),0.9399 ± 0.0019 (med 0.9406),0.9139 ± 0.0025 (med 0.9149),5
3,SNP+MET+CNV+RNA,attention,molformer,0.9688 ± 0.0083 (med 0.9646),0.7171 ± 0.0060 (med 0.7179),0.8826 ± 0.0029 (med 0.8836),0.9398 ± 0.0015 (med 0.9404),0.9139 ± 0.0017 (med 0.9141),5
4,SNP+MET+CNV+RNA,attention,chemberta,0.9693 ± 0.0150 (med 0.9669),0.7216 ± 0.0104 (med 0.7187),0.8824 ± 0.0046 (med 0.8838),0.9396 ± 0.0024 (med 0.9403),0.9134 ± 0.0027 (med 0.9131),5
5,MET+RNA,attention,morgan,0.9700 ± 0.0122 (med 0.9640),0.7206 ± 0.0052 (med 0.7182),0.8822 ± 0.0040 (med 0.8846),0.9395 ± 0.0022 (med 0.9408),0.9129 ± 0.0022 (med 0.9141),5
6,SNP+MET+RNA,attention,morgan,0.9701 ± 0.0049 (med 0.9701),0.7201 ± 0.0080 (med 0.7234),0.8823 ± 0.0011 (med 0.8830),0.9394 ± 0.0006 (med 0.9397),0.9132 ± 0.0011 (med 0.9138),5
7,SNP+RNA,attention,morgan,0.9709 ± 0.0100 (med 0.9705),0.7183 ± 0.0038 (med 0.7177),0.8820 ± 0.0034 (med 0.8830),0.9393 ± 0.0018 (med 0.9399),0.9131 ± 0.0019 (med 0.9121),5
8,SNP+MET+CNV+RNA,attention,graphormer,0.9722 ± 0.0024 (med 0.9713),0.7218 ± 0.0039 (med 0.7213),0.8817 ± 0.0015 (med 0.8822),0.9393 ± 0.0009 (med 0.9397),0.9131 ± 0.0007 (med 0.9131),5
9,SNP+CNV+RNA,attention,morgan,0.9739 ± 0.0144 (med 0.9742),0.7235 ± 0.0119 (med 0.7202),0.8813 ± 0.0042 (med 0.8825),0.9390 ± 0.0022 (med 0.9397),0.9126 ± 0.0026 (med 0.9137),5



===== split_mode = unseen_cell  (sorted by RMSE, lower = better) =====


,omics,cell,drug,rmse,mae,r2,pearson,spearman,n_folds
0,SNP+MET+CNV+RNA,attention,morgan,1.3085 ± 0.0362 (med 1.3128),0.9798 ± 0.0270 (med 0.9836),0.7854 ± 0.0082 (med 0.7845),0.8869 ± 0.0044 (med 0.8865),0.8432 ± 0.0048 (med 0.8430),4


## Ablation views (baseline vs. each axis)
The baseline is `omics=SNP+MET+CNV+RNA, cell=attention, drug=morgan`. Each table
below fixes the other axes at baseline and varies one axis, within a split regime.

In [5]:
BASE = dict(omics='SNP+MET+CNV+RNA', cell='attention', drug='morgan')

def axis_view(sm, vary):
    fixed = {k: v for k, v in BASE.items() if k != vary}
    sub = agg[agg['split_mode'] == sm].copy()
    for k, v in fixed.items():
        sub = sub[sub[k] == v]
    return pretty(sub.sort_values('rmse_mean'))

for sm in ['mixed', 'unseen_cell', 'unseen_drug']:
    if sm not in agg['split_mode'].values:
        continue
    for vary, title in [('omics', 'FEATURE (omics)'), ('cell', 'CELL ENCODER'), ('drug', 'DRUG REP')]:
        v = axis_view(sm, vary)
        if len(v) <= 1:
            continue
        print(f'\n--- {sm} | {title} ablation ---')
        display(v)


--- mixed | FEATURE (omics) ablation ---


,omics,cell,drug,rmse,mae,r2,pearson,spearman,n_folds
0,SNP+MET+CNV+RNA,attention,morgan,0.9652 ± 0.0235 (med 0.9644),0.7156 ± 0.0177 (med 0.7191),0.8834 ± 0.0065 (med 0.8848),0.9401 ± 0.0035 (med 0.9411),0.9143 ± 0.0038 (med 0.9159),5
1,CNV+RNA,attention,morgan,0.9662 ± 0.0116 (med 0.9717),0.7160 ± 0.0088 (med 0.7183),0.8832 ± 0.0038 (med 0.8820),0.9399 ± 0.0020 (med 0.9393),0.9139 ± 0.0019 (med 0.9130),5
2,MET+RNA,attention,morgan,0.9700 ± 0.0122 (med 0.9640),0.7206 ± 0.0052 (med 0.7182),0.8822 ± 0.0040 (med 0.8846),0.9395 ± 0.0022 (med 0.9408),0.9129 ± 0.0022 (med 0.9141),5
3,SNP+MET+RNA,attention,morgan,0.9701 ± 0.0049 (med 0.9701),0.7201 ± 0.0080 (med 0.7234),0.8823 ± 0.0011 (med 0.8830),0.9394 ± 0.0006 (med 0.9397),0.9132 ± 0.0011 (med 0.9138),5
4,SNP+RNA,attention,morgan,0.9709 ± 0.0100 (med 0.9705),0.7183 ± 0.0038 (med 0.7177),0.8820 ± 0.0034 (med 0.8830),0.9393 ± 0.0018 (med 0.9399),0.9131 ± 0.0019 (med 0.9121),5
5,SNP+CNV+RNA,attention,morgan,0.9739 ± 0.0144 (med 0.9742),0.7235 ± 0.0119 (med 0.7202),0.8813 ± 0.0042 (med 0.8825),0.9390 ± 0.0022 (med 0.9397),0.9126 ± 0.0026 (med 0.9137),5
6,MET+CNV+RNA,attention,morgan,0.9747 ± 0.0155 (med 0.9812),0.7227 ± 0.0137 (med 0.7242),0.8811 ± 0.0047 (med 0.8797),0.9389 ± 0.0025 (med 0.9382),0.9124 ± 0.0029 (med 0.9115),5



--- mixed | CELL ENCODER ablation ---


,omics,cell,drug,rmse,mae,r2,pearson,spearman,n_folds
0,SNP+MET+CNV+RNA,attention,morgan,0.9652 ± 0.0235 (med 0.9644),0.7156 ± 0.0177 (med 0.7191),0.8834 ± 0.0065 (med 0.8848),0.9401 ± 0.0035 (med 0.9411),0.9143 ± 0.0038 (med 0.9159),5
1,SNP+MET+CNV+RNA,mlp,morgan,0.9934 ± 0.0183 (med 0.9923),0.7370 ± 0.0124 (med 0.7338),0.8765 ± 0.0052 (med 0.8760),0.9364 ± 0.0028 (med 0.9362),0.9086 ± 0.0034 (med 0.9092),5



--- mixed | DRUG REP ablation ---


,omics,cell,drug,rmse,mae,r2,pearson,spearman,n_folds
0,SNP+MET+CNV+RNA,attention,morgan,0.9652 ± 0.0235 (med 0.9644),0.7156 ± 0.0177 (med 0.7191),0.8834 ± 0.0065 (med 0.8848),0.9401 ± 0.0035 (med 0.9411),0.9143 ± 0.0038 (med 0.9159),5
1,SNP+MET+CNV+RNA,attention,unimol,0.9671 ± 0.0118 (med 0.9621),0.7187 ± 0.0099 (med 0.7188),0.8830 ± 0.0033 (med 0.8843),0.9399 ± 0.0019 (med 0.9406),0.9139 ± 0.0025 (med 0.9149),5
2,SNP+MET+CNV+RNA,attention,molformer,0.9688 ± 0.0083 (med 0.9646),0.7171 ± 0.0060 (med 0.7179),0.8826 ± 0.0029 (med 0.8836),0.9398 ± 0.0015 (med 0.9404),0.9139 ± 0.0017 (med 0.9141),5
3,SNP+MET+CNV+RNA,attention,chemberta,0.9693 ± 0.0150 (med 0.9669),0.7216 ± 0.0104 (med 0.7187),0.8824 ± 0.0046 (med 0.8838),0.9396 ± 0.0024 (med 0.9403),0.9134 ± 0.0027 (med 0.9131),5
4,SNP+MET+CNV+RNA,attention,graphormer,0.9722 ± 0.0024 (med 0.9713),0.7218 ± 0.0039 (med 0.7213),0.8817 ± 0.0015 (med 0.8822),0.9393 ± 0.0009 (med 0.9397),0.9131 ± 0.0007 (med 0.9131),5


## Save aggregated tables

In [6]:
os.makedirs('outputs', exist_ok=True)
folds.to_csv('outputs/cv_fold_records.csv', index=False)
agg.round(6).to_csv('outputs/cv_summary_by_condition.csv', index=False)
print('saved:', os.path.abspath('outputs/cv_summary_by_condition.csv'))
print('      ', os.path.abspath('outputs/cv_fold_records.csv'))

saved: /project/OmicsDRP_Review/omicsdrp/notebooks/outputs/cv_summary_by_condition.csv
       /project/OmicsDRP_Review/omicsdrp/notebooks/outputs/cv_fold_records.csv
